In [2]:
import json
import gurobipy as gp
from gurobipy import GRB
import numpy as np
import matplotlib.pyplot as plt
import sympy as sp
from scipy.special import stirling2
import math
from copy import copy

In [3]:
status_codes = {
    1: 'LOADED',
    2: 'OPTIMAL',
    3: 'INFEASIBLE',
    4: 'INF_OR_UNBD',
    5: 'UNBOUNDED',
    6: 'CUTOFF',
    7: 'ITERATION_LIMIT',
    8: 'NODE_LIMIT',
    9: 'TIME_LIMIT',
    10: 'SOLUTION_LIMIT',
    11: 'INTERRUPTED',
    12: 'NUMERIC',
    13: 'SUBOPTIMAL',
    14: 'INPROGRESS',
    15: 'USER_OBJ_LIMIT'
}

In [4]:
rng = np.random.default_rng(643)

# SDP Minimal Working Example

# Multivariate utils

In [20]:
def compute_order(alpha):
    '''Sum of elements of a power.'''
    order = 0
    for alpha_i in alpha:
        order += alpha_i
    return order

def compute_Nd(S, d):
    '''Number of moments of order <= d (S species)'''
    Nd = math.factorial(S + d) // (math.factorial(d) * math.factorial(S))
    return Nd

def compute_powers(S, d):
    '''Compute the Nd powers of order <= d (S species)'''

    # all powers
    powers = [[0 for s in range(S)]]

    # powers of order d = 0
    powers_prev = [[0 for s in range(S)]]

    # for order d = 1, ..., d
    for order in range(1, d + 1):

        # store powers of order d
        powers_current = []

        # for each power of order d - 1
        for alpha in powers_prev:

            # for each index
            for i in range(S):

                # add 1 to power at index
                alpha_new = copy(alpha)
                alpha_new[i] += 1

                # store (avoid repeats)
                if alpha_new not in powers_current:
                    powers_current.append(alpha_new)

        # update d - 1 powers
        powers += powers_current

        # update overall powers
        powers_prev = powers_current

    return powers

def add_powers(*powers, S):
    '''Add powers (lists) of size S.'''
    plus = [0 for i in range(S)]
    for i in range(S):
        for power in powers:
            plus[i] += power[i]
    return plus

def falling_factorial(n, k):
    '''FF(n,k) = n(n-1) ... (n - k + 1).'''
    val = 1
    for i in range(k):
        val *= (n - i)
    return val

def binomial_moment(n, p, l):
    '''For X ~ Bin(n, p) compute E[X^l].'''
    val = 0
    for k in range(l + 1):
        val += falling_factorial(n, k) * stirling2(l, k) * p**k
    return val

# Bootstrap

In [21]:
def bootstrap(sample, d, resamples=1000):
    '''
    Compute confidence intervals on the moments of a sample of count pairs, use
    resamples number of bootstrap resamples (default to sample size) and estimate
    moments up to order d.

    Args:
        sample: cell x observed species (n x SO) array of integer counts
        d: maximum moment order to estimate
        resamples: integer number of bootstrap resamples to use

    Returns:
        (2 x Nd) numpy array of CI bounds on each Nd moment of order <= d
    '''

    # size
    n, SO = sample.shape 

    # helpful values
    powers = compute_powers(S=SO, d=d)
    Nd = compute_Nd(S=SO, d=d)

    # bootstrap to N x n x SO array
    boot = rng.choice(sample, size=(resamples, n))

    # split into 2 N x n arrays
    x1_boot = boot[:, :, 0]
    x2_boot = boot[:, :, 1]

    # estimate
    moment_bounds = np.zeros((2, Nd))
    for i, alpha in enumerate(powers):

        # compute x1**alpha1 * ... * xSO**alphaSO over bootstrap
        boot_alpha = 1

        # for each species
        for j, alpha_j in enumerate(alpha):

            # raise to power
            xj_boot_alpha = boot[:, :, j]**alpha[j]

            # multiply
            boot_alpha *= xj_boot_alpha

        # mean over sample axis (N x n) -> (N x 1)
        moment_estimates = np.mean(boot_alpha, axis=1)

        # quantile over boot axis (N x 1) -> (2 x 1)
        moment_interval = np.quantile(moment_estimates, [0.025, 0.975])

        # store
        moment_bounds[:, i] = moment_interval

    return moment_bounds

# Downsampling

In [22]:
def downsample_data(sample, mean_capture):

    # size
    n, _ = sample.shape

    # parameters
    b = (1 / mean_capture) - 1

    # capture efficiency
    if b == 0:
        beta = np.ones(n)
    else:
        beta = rng.beta(1, b, size=n)
    
    # downsample
    downsample = rng.binomial(sample, beta[:, None])

    return downsample, beta

# Moment equations

In [23]:
def compute_A(alpha, reactions, vrs, db, R, S, d):
    '''
    Moment equation coefficient matrix
    NOTE: must have order of alpha <= d

    Args:
        alpha: moment order for equation (d/dt mu^alpha = 0)
        reactions: list of strings detailing a_r(x) for each reaction r
        vrs: list of lists detailing v_r for each reaction r
        db: largest order a_r(x)
        R: number of reactions
        S: number of species
        d: maximum moment order used (must be >= order(alpha) + db - 1)

    Returns:
        A: (R, Nd) matrix of coefficients
    '''

    if compute_order(alpha) > d - db + 1:
        raise NotImplementedError(f"Maximum moment order {d} too small for moment equation of alpha = {alpha}: involves moments of higher order.")

    xs = sp.symbols([f'x{i}' for i in range(S)])

    # reaction propensity polynomials
    # props = [eval(str_ar) for str_ar in reactions]
    props = [sp.parse_expr(str_ar, {'xs': xs}) for str_ar in reactions]

    # number of moments of order <= d
    Nd = compute_Nd(S, d)

    # get powers of order <= d
    powers = compute_powers(S, d)

    # setup matrix
    A = np.zeros((R, Nd))

    for r, prop in enumerate(props):

        # expand b(x) * ((x + v_r)**alpha - x**alpha)
        term_1 = 1
        term_2 = 1
        for i in range(S):
            term_1 = term_1 * (xs[i] + vrs[r][i])**alpha[i]
            term_2 = term_2 * xs[i]**alpha[i]
        poly = sp.Poly(prop * (term_1 - term_2), xs)

        # loop over terms
        for xs_power, coeff in zip(poly.monoms(), poly.coeffs()):

            # get matrix index
            col = powers.index(list(xs_power))

            # store
            A[r, col] = coeff

    return A

# B Capture matrix

In [24]:
def compute_B(beta, S, U, d):
    '''
    Capture efficiency moment scaling matrix

    Args:
        beta: per cell capture efficiency sample
        S: number of species
        U: unobserved species indices
        d: maximum moment order used

    Returns:
        B: (Nd, Nd) matrix of coefficients
    '''

    # number of moments of order <= d
    Nd = compute_Nd(S, d)

    # compute powers of order <= d
    powers = compute_powers(S, d)

    # compute beta moments of order <= d
    y_beta = np.zeros(d + 1)
    for l in range(d + 1):
        y_beta[l] = np.mean(beta**l)

    # setup matrix
    B = np.zeros((Nd, Nd))

    p = sp.Symbol('p')
    xs = sp.symbols([f'x{i}' for i in range(S)])

    # for each moment power
    for row, alpha in enumerate(powers):

        # setup polynomail
        poly_alpha = 1

        # for each species
        for i in range(S):

            # unobserved: no capture efficiency
            if i in U:
                moment = xs[i]**alpha[i]

            # observed: compute moment expression for E[Xi^alphai] in xi
            else:
                moment = binomial_moment(xs[i], p, alpha[i])
            
            poly = sp.Poly(moment, p, xs[i])

            # multiply
            poly_alpha = poly_alpha * poly

        # loop over terms
        for (beta_power, *xs_power), coeff in zip(poly_alpha.monoms(), poly_alpha.coeffs()):

            # get matrix index
            col = powers.index(xs_power)

            B[row, col] += coeff * y_beta[beta_power]

    return B

# Moment matrices

In [25]:
def construct_M_s(y, s, S, d):
    '''Moment matrix variable constructor (s).'''
    if s == 0:
        D = math.floor(d / 2)
    else:
        D = math.floor((d - 1) / 2)
    powers_D = compute_powers(S, D)
    powers_d = compute_powers(S, d)
    ND = compute_Nd(S, D)
    M_s = [[0 for j in range(ND)] for i in range(ND)]
    e_s = [1 if i == (s - 1) else 0 for i in range(S)]
    for alpha_index, alpha in enumerate(powers_D):
        for beta_index, beta in enumerate(powers_D):
            plus = add_powers(alpha, beta, e_s, S=S)
            plus_index = powers_d.index(plus)
            M_s[alpha_index][beta_index] = y[plus_index].item()
    M_s = gp.MVar.fromlist(M_s)
    return M_s

# NLP base model

In [254]:
def base_model(model, OB_bounds, beta, reactions, vrs, db, R, S, U, d, d_bd, d_me, d_sd, constraints, fixed=[], fixed_weak=[], time_limit=300, K=100):
    '''
    Construct 'base model' with semidefinite constraints removed to give NLP

    Args:
        opt: Optimization class (or subclass), see relevant attributes
        model: empty gurobi model object
        OB_bounds: confidence intervals on observed moments up to order d (at least)

        Relevant class attributes

        beta: capture efficiency vector
        reactions: list of strings detailing a_r(x) for each reaction r
        vrs: list of lists detailing v_r for each reaction r
        db: largest order a_r(x)
        R: number of reactions
        S: number of species
        U: indices of unobserved species
        d: maximum moment order used
        fixed: list of pairs of (reaction index r, value to fix k_r to)
        time_limit: optimization time limit

        constraint options

        moment_bounds: CI bounds on moments
        moment_matrices: 
        moment_equations
        factorization
        factorization_telegraph
        telegraph_moments

    Returns:
        model: gurobi model object with NLP constraints (all but semidefinite)
        variables: dict for model variable reference
    '''

    # model settings
    model.Params.TimeLimit = time_limit

    # helpful values
    Nd = compute_Nd(S, d)
    O = [i for i in range(S) if i not in U]
    SO = len(O)

    # variables
    y = model.addMVar(shape=Nd, vtype=GRB.CONTINUOUS, name="y", lb=0)
    k = model.addMVar(shape=R, vtype=GRB.CONTINUOUS, name="k", lb=0, ub=K)

    # variable dict
    variables = {
        'y': y,
        'k': k
    }

    if constraints['moment_matrices']:

        # moment matrices
        for s in range(S + 1):
            M_s = construct_M_s(y, s, S, d_sd)
            variables[f'M_{s}'] = M_s
    
    # constraints

    if constraints['moment_bounds']:

        # only explicitly bound observed, leave unobserved unbounded
        # avoids issues with e+100 upper bounds on unobserved moments

        # B scaling matrix
        B = compute_B(beta, S, U, d)

        # downsampled moments
        y_D = B @ y

        # powers up to order d_bd for all species & observed species
        powers_S = compute_powers(S, d_bd)
        powers_SO = compute_powers(SO, d_bd)

        # for all species power
        for i, alpha_S in enumerate(powers_S):

            # skip if contains unobserved species (non-zero power)
            unobserved = False
            for j in U:
                if alpha_S[j] > 0:
                    unobserved = True
            if unobserved:
                continue

            # otherwise: bound

            # find corresponding index of observed species power
            alpha_SO = [alpha_S[i] for i in O]
            j = powers_SO.index(alpha_SO)
    
            # bound
            model.addConstr(y_D[i] <= OB_bounds[1, j], name=f"y_{i}_UB")
            model.addConstr(y_D[i] >= OB_bounds[0, j], name=f"y_{i}_LB")

    if constraints['moment_equations']:
                   
        # moment equations (order(alpha) <= d - db + 1)
        moment_powers = compute_powers(S, d_me - db + 1)
        for alpha in moment_powers:
            A_alpha_d = compute_A(alpha, reactions, vrs, db, R, S, d)
            model.addConstr(k.T @ A_alpha_d @ y == 0, name=f"ME_{alpha}_{d}")

    if constraints['unobserved_constraints']:

        # for each power
        powers = compute_powers(S, d)
        for i, alpha in enumerate(powers):

            # for each unobserved species
            for j in U:

                # if unobserved species has power >= 1
                if alpha[j] > 1:

                    # set equal to moment with power = 1
                    alpha_reduced = alpha
                    alpha_reduced[j] = 1
                    l = powers.index(alpha_reduced)
                    model.addConstr(y[i] == y[l], name="U_eq")

                # if unobserved species has power 1
                elif alpha[j] == 1:

                    # constrain to less than moment with power 0
                    alpha_reduced = alpha
                    alpha_reduced[j] = 0
                    l = powers.index(alpha_reduced)
                    model.addConstr(y[i] <= y[l], name="U_ineq")

    # fixed moment
    model.addConstr(y[0] == 1, name="y0_base")

    # fixed parameters
    for r, val in fixed:
        model.addConstr(k[r] == val, name=f"k{r}_fixed")
    for r, val in fixed_weak:
        model.addConstr(k[r] >= val, name=f"k{r}_fixed_weak")

    return model, variables

In [27]:
def optimize(model, obj):
    '''Optimize model, return status.'''

    # optimize
    model.setObjective(obj, GRB.MINIMIZE)
    model.optimize()
    status = status_codes[model.status]

    # get variable values
    all_vars = model.getVars()
    try:
        values = model.getAttr("X", all_vars)
    except:
        values = [None for var in all_vars]
    names = model.getAttr("VarName", all_vars)
    var_dict = {name: val for name, val in zip(names, values)}

    return model, status, var_dict

# Semidefinite check & cuttting planes

In [28]:
def semidefinite_cut(model, variables, S, print_evals=False, eval_eps=10**-6, printing=False):
    '''
    Check semidefinite feasibility of NLP feasible point
    Feasible: stop
    Infeasible: add cutting plane (ALL negative eigenvalues)

    Args:
        model: optimized NLP model
        variables: model variable reference dict
        print_evals: option to display moment matrix eigenvalues (semidefinite condition)

    Returns:
        model: model with any cutting planes added
        bool: semidefinite feasibility status
    '''

    # data list
    data = []

    # moment matrix values
    for s in range(S + 1):
        data.append(
            {f'M_val': variables[f'M_{s}'].X}
        )

    # eigen information
    for s in range(S + 1):
        evals_s, evecs_s = np.linalg.eigh(data[s]['M_val'])
        data[s]['evals'] = evals_s
        data[s]['evecs'] = evecs_s

    # extract eigenvalue data
    evals_data = {s: data[s]['evals'] for s in range(S + 1)}

    if print_evals:
        print("Moment matices eigenvalues:")
        for s in range(S + 1):
            print(data[s]['evals'])

    # check if all positive eigenvalues
    positive = True
    for s in range(S + 1):
        if not (data[s]['evals'] >= -eval_eps).all():
            positive = False
            break

    # positive eigenvalues
    if positive:

        if printing: print("SDP feasible\n")
    
        return model, True, evals_data

    # negative eigenvalue
    else:

        if printing: print("SDP infeasible\n")

        # for each matrix
        for s in range(S + 1):

            # for each M_s eigenvalue
            for i, lam in enumerate(data[s]['evals']):

                # if negative (sufficiently)
                if lam < -eval_eps:

                    # get evector
                    v = data[s]['evecs'][:, i]

                    # add cutting plane
                    #model.addConstr(np.kron(v, v.T) @ variables[f'M_{s}'].reshape(-1) >= 0, name=f"Cut_{s}")
                    model.addConstr(v.T @ variables[f'M_{s}'] @ v >= 0, name=f"Cut_{s}")
                
                    if printing: print(f"M_{s} cut added")

        if printing: print("")

    return model, False, evals_data

# SDP feasibility algorithm

In [255]:
def feasibility_test(OB_bounds, beta, reactions, vrs, db, R, S, U, d, d_bd, d_me, d_sd, obj_index, constraints, fixed=[], fixed_weak=[],
                     time_limit=300, K=100, print_evals=False, eval_eps=10**-6, printing=False,
                     silent=True, write_model=False, load_model=False, cut_limit=100, total_time_limit=300):
    '''
    Full feasibility test of birth death model via following algorithm

    Optimize NLP
    Infeasible: stop
    Feasible: check SDP feasibility
        Feasible: stop
        Infeasible: add cutting plane and return to NLP step
    '''

    # store information from SDP loop
    eigenvalues = []
    optim_times = []
    feasible_values = []

    # silence output
    environment_parameters = {}
    if silent:
        environment_parameters['OutputFlag'] = 0

    # environment context
    with gp.Env(params=environment_parameters) as env:

        # model context
        with gp.Model('test-SDP', env=env) as model:

             # if provided: load model
            if load_model:

                # get model
                model = gp.read(load_model, env)

                # get variables
                Nd = compute_Nd(S, d)
                variables = {
                    'y': gp.MVar([model.getVarByName(f'y[{i}]') for i in range(Nd)]),
                    'k': gp.MVar([model.getVarByName(f'k[{i}]') for i in range(R)])
                }
                if constraints['moment_matrices']:
                    for s in range(S + 1):
                        M_s = construct_M_s(variables['y'], s, S, d_sd)
                        variables[f'M_{s}'] = M_s

                # general setup
                model.Params.TimeLimit = time_limit

            # otherwise: construct base model (no semidefinite constraints)
            else:
                model, variables = base_model(model, OB_bounds, beta, reactions, vrs, db, R, S, U, d, d_bd, d_me, d_sd, constraints, fixed=fixed, fixed_weak=fixed_weak, time_limit=time_limit, K=K)

            # choose objective
            if obj_index == -1:
                obj = 0
            else:
                obj = variables['k'][obj_index]
            
            # check feasibility
            model, status, var_dict = optimize(model, obj)

            # collect solution information
            solution = {
                'status': status,
                'time': model.Runtime,
                'cuts': 0
            }
            optim_times.append(solution['time'])
            feasible_values.append(var_dict)

            # no semidefinite constraints or non-optimal solution: return NLP status
            if not (constraints['moment_matrices'] and status == "OPTIMAL"):

                # write model
                if write_model:
                    model.write('model.lp')

                return solution, eigenvalues, optim_times, feasible_values

            # while below time and cut limit
            while (solution['cuts'] < cut_limit) and (solution['time'] < total_time_limit):

                # check semidefinite feasibility & add cuts if needed
                model, semidefinite_feas, evals_data = semidefinite_cut(model, variables, S, print_evals, eval_eps, printing)

                # store eigenvalue
                eigenvalues.append(evals_data)

                # semidefinite feasible: return
                if semidefinite_feas:

                    # write model
                    if write_model:
                        model.write('model.lp')

                    return solution, eigenvalues, optim_times, feasible_values
                
                # record cut
                solution['cuts'] += 1
                
                # semidefinite infeasible: check NLP feasibility with added cut
                model, status, var_dict = optimize(model, obj)

                # update optimization time
                solution['time'] += model.Runtime

                # store feasible values & optim times
                feasible_values.append(var_dict)
                optim_times.append(model.Runtime)

                # NLP + cut infeasible: return
                # (also return for any other status, can only proceed if optimal as need feasible point)
                if not (status == "OPTIMAL"):

                    # update solution
                    solution['status'] = status

                    # write model
                    if write_model:
                        model.write('model.lp')

                    return solution, eigenvalues, optim_times, feasible_values

            # set custom status
            if solution['cuts'] >= cut_limit:

                # exceeded number of cutting plane iterations
                solution['status'] = "CUT_LIMIT"
            
            elif solution['time'] >= total_time_limit:

                # exceeded total optimization time
                solution['status'] = "TOTAL_TIME_LIMIT"

            # write model
            if write_model:
                model.write('model.lp')

            return solution, eigenvalues, optim_times, feasible_values

# Compute feasible correlation

In [ ]:
def compute_feasible_correlation(S, d, var_dict, ix, iy):
    '''Compute correlation value at feasible point.'''

    def ei(*idxs, val=1):
        power = np.zeros(S)
        for i in idxs:
            power[i] = val
        return list(power)
    
    # find indices of moments
    powers = compute_powers(S, d)
    i_xy = powers.index(ei(ix, iy))
    i_x = powers.index(ei(ix))
    i_y = powers.index(ei(iy))
    i_x2 = powers.index(ei(ix, val=2))
    i_y2 = powers.index(ei(iy, val=2))

    # collect moment values
    E_xy = var_dict[f'y[{i_xy}]']
    E_x  = var_dict[f'y[{i_x}]']
    E_y  = var_dict[f'y[{i_y}]']
    E_x2 = var_dict[f'y[{i_x2}]']
    E_y2 = var_dict[f'y[{i_y2}]']

    # compute statistics
    cov_xy = E_xy - E_x*E_y
    var_x = E_x2 - E_x**2
    var_y = E_y2 - E_y**2

    # return None if correlation undefined
    if var_x <= 0 or var_y <= 0:
        return None

    # compute correlation
    correlation = cov_xy / (np.sqrt(var_x) * np.sqrt(var_y))

    return correlation

# Birth Death

In [31]:
from SDP_interaction_inference.simulation import gillespie_birth_death

In [32]:
# settings
k_tx = 5
k_deg = 1
k_reg = 2

n = 1000
N = 1000

# sample
params = {
    'k_tx_1': k_tx,
    'k_tx_2': k_tx,
    'k_deg_1': k_deg,
    'k_deg_2': k_deg,
    'k_reg': k_reg
}
sample = gillespie_birth_death(params, n)

In [35]:
sample = np.array(sample)

In [36]:
# downsample
mean_capture = 1.0
downsample, beta = downsample_data(sample, mean_capture)

# mean expression level
print(f"Mean expression {np.mean(downsample)}")

Mean expression 1.63


In [37]:
# settings
reactions = [
    "1",
    "xs[0]",
    "1",
    "xs[1]",
    "xs[0] * xs[1]"
]
vrs = [
    [1, 0],
    [-1, 0],
    [0, 1],
    [0, -1],
    [-1, -1]
]
db = 2
R = 5
S = 2
U = []

d = 3
d_bd = 3
d_me = 3
d_sd = 3

K = 100
fixed = [(3, 1)]
obj_index = 4

# constraints
constraints = {
    'moment_bounds':           True,
    'moment_matrices':         True,
    'moment_equations':        True,
    'unobserved_constraints':  False
}

# bootstrap
OB_bounds = bootstrap(downsample, d, N)

# test feasibility
solution, eigenvalues, optim_times, feasible_values = feasibility_test(
    OB_bounds,
    beta,
    reactions,
    vrs,
    db,
    R,
    S,
    U, 
    d,
    d_bd,
    d_me,
    d_sd,
    obj_index,
    constraints,
    fixed=fixed,
    time_limit=300,
    K=K,
    print_evals=False,
    eval_eps=10**-6,
    printing=True,
    silent=True,
    write_model=False,
    load_model=False,
    cut_limit=100,
    total_time_limit=300
)

SDP feasible



In [38]:
solution

{'status': 'OPTIMAL', 'time': 0.10000014305114746, 'cuts': 0}

In [39]:
feasible_values[-1][f'k[{obj_index}]']

0.3787122054805344

In [40]:
[feasible_values[-1][f'k[{r}]'] for r in range(R)]

[0.7252338734952234, 0.0, 2.2993621591829005, 1.0, 0.3787122054805344]

In [ ]:
# wrong: ix, iy error
compute_feasible_correlation(S, d, feasible_values[-1], 0, 1)

np.float64(-0.3110355202431624)

In [44]:
np.corrcoef(sample.T)

array([[ 1.        , -0.38479421],
       [-0.38479421,  1.        ]])

# Telegraph

In [45]:
from SDP_interaction_inference.simulation import gillespie_telegraph

In [46]:
# settings
k_on = 0.75
k_off = 0.75
k_tx = 5
k_deg = 1
k_reg = 5

n = 1000
N = 1000

# sample
params = {
    'k_on_1': k_on,
    'k_on_2': k_on,
    'k_off_1': k_off,
    'k_off_2': k_off,
    'k_tx_1': k_tx,
    'k_tx_2': k_tx,
    'k_deg_1': k_deg,
    'k_deg_2': k_deg,
    'k_reg': k_reg
}
sample = gillespie_telegraph(params, n)

In [47]:
sample = np.array(sample)

In [48]:
# downsample
mean_capture = 1.0
downsample, beta = downsample_data(sample, mean_capture)

# mean expression level
print(f"Mean expression {np.mean(downsample)}")

Mean expression 1.238


In [51]:
# settings
reactions = [
    "1 - xs[2]",
    "xs[2]",
    "xs[2]",
    "xs[0]",
    "1 - xs[3]",
    "xs[3]",
    "xs[3]",
    "xs[1]",
    "xs[0] * xs[1]"
]
vrs = [
    [0, 0, 1, 0],
    [0, 0, -1, 0],
    [1, 0, 0, 0],
    [-1, 0, 0, 0],
    [0, 0, 0, 1],
    [0, 0, 0, -1],
    [0, 1, 0, 0],
    [0, -1, 0, 0],
    [-1, -1, 0, 0]
]
db = 2
R = 9
S = 4
U = [2, 3]

d = 3
d_bd = 3
d_me = 3
d_sd = 3

K = 100
fixed = [(3, 1)]
obj_index = 8

# constraints
constraints = {
    'moment_bounds':           True,
    'moment_matrices':         True,
    'moment_equations':        True,
    'unobserved_constraints':  True,
}

# bootstrap
OB_bounds = bootstrap(downsample, d, N)

# test feasibility
solution, eigenvalues, optim_times, feasible_values = feasibility_test(
    OB_bounds,
    beta,
    reactions,
    vrs,
    db,
    R,
    S,
    U, 
    d,
    d_bd,
    d_me,
    d_sd,
    obj_index,
    constraints,
    fixed=fixed,
    time_limit=300,
    K=K,
    print_evals=False,
    eval_eps=10**-6,
    printing=True,
    silent=True,
    write_model=False,
    load_model=False,
    cut_limit=100,
    total_time_limit=300
)

SDP infeasible

M_1 cut added
M_2 cut added
M_3 cut added

SDP infeasible

M_1 cut added
M_3 cut added
M_3 cut added

SDP infeasible

M_2 cut added
M_3 cut added
M_3 cut added

SDP infeasible

M_1 cut added
M_2 cut added
M_3 cut added

SDP infeasible

M_1 cut added
M_2 cut added
M_3 cut added

SDP infeasible

M_1 cut added
M_2 cut added

SDP infeasible

M_1 cut added
M_2 cut added
M_3 cut added

SDP infeasible

M_2 cut added
M_3 cut added
M_3 cut added

SDP infeasible

M_3 cut added

SDP infeasible

M_1 cut added
M_3 cut added

SDP infeasible

M_1 cut added
M_3 cut added

SDP infeasible

M_3 cut added

SDP infeasible

M_2 cut added
M_3 cut added

SDP infeasible

M_2 cut added
M_3 cut added

SDP infeasible

M_2 cut added
M_3 cut added

SDP infeasible

M_2 cut added
M_3 cut added

SDP infeasible

M_2 cut added
M_3 cut added

SDP infeasible

M_2 cut added
M_3 cut added

SDP infeasible

M_2 cut added
M_3 cut added

SDP infeasible

M_2 cut added
M_3 cut added

SDP infeasible

M_2 cut added


In [52]:
solution

{'status': 'OPTIMAL', 'time': 0.5790002346038818, 'cuts': 21}

In [53]:
feasible_values[-1][f'k[{obj_index}]']

0.0

In [54]:
[feasible_values[-1][f'k[{r}]'] for r in range(R)]

[0.0, 0.0, 4.2058929408719266, 1.0, 0.0, 0.0, 0.149198102209089, 0.0, 0.0]

In [ ]:
# wrong: ix, iy error
compute_feasible_correlation(S, d, feasible_values[-1], 0, 1)

np.float64(-0.38000771310449993)

In [56]:
np.corrcoef(sample.T)

array([[ 1.        , -0.40116549],
       [-0.40116549,  1.        ]])

# Multiple Birth Death

In [202]:
# settings
Sigma = np.array([
    [1, 0.5, -0.5],
    [-0.5, 1, 0],
    [-0.5, 0, 1]]
)
A = np.linalg.cholesky(Sigma)
b = np.array([0.5, 0.5, 0.5])
G = 3

n = 1000
N = 1000

# sample
z = rng.normal(size=(n, G))
lam = np.exp(z @ A.T + b)
#sample = rng.poisson(lam)

In [203]:
# settings
k_tx = 5
k_deg = 1
k_reg = 0

n = 1000
N = 1000

# sample
sample_x1 = rng.poisson(k_tx / k_deg, size=n)
sample_x2 = rng.poisson(k_tx / k_deg, size=n)
sample_x3 = rng.poisson(k_tx / k_deg, size=n)
sample = np.vstack([sample_x1, sample_x2, sample_x3]).T

In [204]:
# downsample
mean_capture = 1.0
downsample, beta = downsample_data(sample, mean_capture)

# mean expression level
print(f"Mean expression {np.mean(downsample)}")

Mean expression 5.017333333333333


In [205]:
# settings
reactions = [
    "1",
    "xs[0]",
    "1",
    "xs[1]",
    "1",
    "xs[2]",
    "xs[0] * xs[1]",
    "xs[0] * xs[2]"
]
vrs = [
    [1, 0, 0],
    [-1, 0, 0],
    [0, 1, 0],
    [0, -1, 0],
    [0, 0, 1],
    [0, 0, -1],
    [-1, -1, 0],
    [-1, 0, -1]
]
db = 2
R = 8
S = 3
U = []

d = 3
d_bd = 3
d_me = 3
d_sd = 3

K = 100
fixed = [(5, 1)]
obj_index = -1 # 6

# constraints
constraints = {
    'moment_bounds':           True,
    'moment_matrices':         True,
    'moment_equations':        True,
    'unobserved_constraints':  False
}

# bootstrap
OB_bounds = bootstrap(downsample, d, N)

# test feasibility
solution, eigenvalues, optim_times, feasible_values = feasibility_test(
    OB_bounds,
    beta,
    reactions,
    vrs,
    db,
    R,
    S,
    U, 
    d,
    d_bd,
    d_me,
    d_sd,
    obj_index,
    constraints,
    fixed=fixed,
    time_limit=300,
    K=K,
    print_evals=False,
    eval_eps=10**-6,
    printing=True,
    silent=True,
    write_model=False,
    load_model=False,
    cut_limit=100,
    total_time_limit=300
)

SDP feasible



In [206]:
solution

{'status': 'OPTIMAL', 'time': 0.016000032424926758, 'cuts': 0}

In [207]:
#feasible_values[-1][f'k[{obj_index}]']

In [208]:
[feasible_values[-1][f'k[{r}]'] for r in range(R)]

[75.84400526639276,
 10.423708098811524,
 59.44707917340308,
 9.117856022986258,
 16.107668753277128,
 1.0,
 0.4902450919287284,
 0.45571484113457483]

In [ ]:
# wrong: ix, iy error
rho_12 = compute_feasible_correlation(S, d, feasible_values[-1], 0, 1)
rho_13 = compute_feasible_correlation(S, d, feasible_values[-1], 0, 2)
rho_23 = compute_feasible_correlation(S, d, feasible_values[-1], 1, 2)
float(rho_12), float(rho_13), float(rho_23)

(-0.002210298927124612, -0.19740737918664958, -0.17782895888298747)

In [210]:
corr = np.corrcoef(sample.T)
float(corr[1, 0]), float(corr[2, 0]), float(corr[2, 1])

(-0.014372194187782785, 0.014466918843306528, 0.0759701194153297)

## Notes

- PLN seems to be infeasible under multiple birth-death model
    - chose parameters so mRNAs independent, miRNA negatively correlated to both

# Multiple Telegraph

In [178]:
# settings
Sigma = np.array([
    [1, 0.5, -0.8],
    [0.5, 1, 0],
    [-0.8, 0, 1]]
)
A = np.linalg.cholesky(Sigma)
b = np.array([0.5, 0.5, 0.5])
G = 3

n = 1000
N = 1000

# sample
z = rng.normal(size=(n, G))
lam = np.exp(z @ A.T + b)
sample = rng.poisson(lam)

In [179]:
# settings
k_tx = 5
k_deg = 1
k_reg = 0

n = 1000
N = 1000

# sample
sample_x1 = rng.poisson(k_tx / k_deg, size=n)
sample_x2 = rng.poisson(k_tx / k_deg, size=n)
sample_x3 = rng.poisson(k_tx / k_deg, size=n)
#sample = np.vstack([sample_x1, sample_x2, sample_x3]).T

In [180]:
# downsample
mean_capture = 1.0
downsample, beta = downsample_data(sample, mean_capture)

# mean expression level
print(f"Mean expression {np.mean(downsample)}")

Mean expression 2.763333333333333


In [181]:
# settings
reactions = [
    "1 - xs[3]",
    "xs[3]",
    "xs[3]",
    "xs[0]",
    "1 - xs[4]",
    "xs[4]",
    "xs[4]",
    "xs[1]",
    "1 - xs[5]",
    "xs[5]",
    "xs[5]",
    "xs[2]",
    "xs[0] * xs[1]",
    "xs[0] * xs[2]"
]
vrs = [
    [0, 0, 0, 1, 0, 0],
    [0, 0, 0, -1, 0, 0],
    [1, 0, 0, 0, 0, 0],
    [-1, 0, 0, 0, 0, 0],
    [0, 0, 0, 0, 1, 0],
    [0, 0, 0, 0, -1, 0],
    [0, 1, 0, 0, 0, 0],
    [0, -1, 0, 0, 0, 0],
    [0, 0, 0, 0, 0, 1],
    [0, 0, 0, 0, 0, -1],
    [0, 0, 1, 0, 0, 0],
    [0, 0, -1, 0, 0, 0],
    [-1, -1, 0, 0, 0, 0],
    [-1, 0, -1, 0, 0, 0]
]
db = 2
R = 14
S = 6
U = [3, 4, 5]

d = 3
d_bd = 3
d_me = 3
d_sd = 3

K = 100
fixed = [(3, 1)]
obj_index = -1 #12

# constraints
constraints = {
    'moment_bounds':           True,
    'moment_matrices':         True,
    'moment_equations':        True,
    'unobserved_constraints':  True,
}

# bootstrap
OB_bounds = bootstrap(downsample, d, N)

# test feasibility
solution, eigenvalues, optim_times, feasible_values = feasibility_test(
    OB_bounds,
    beta,
    reactions,
    vrs,
    db,
    R,
    S,
    U, 
    d,
    d_bd,
    d_me,
    d_sd,
    obj_index,
    constraints,
    fixed=fixed,
    time_limit=300,
    K=K,
    print_evals=False,
    eval_eps=10**-6,
    printing=True,
    silent=True,
    write_model=False,
    load_model=False,
    cut_limit=100,
    total_time_limit=300
)

SDP infeasible

M_1 cut added
M_2 cut added
M_3 cut added
M_4 cut added

SDP infeasible

M_2 cut added
M_4 cut added
M_4 cut added

SDP infeasible

M_2 cut added
M_3 cut added
M_4 cut added
M_4 cut added

SDP infeasible

M_1 cut added
M_2 cut added
M_3 cut added
M_4 cut added

SDP infeasible

M_1 cut added
M_3 cut added
M_4 cut added

SDP infeasible

M_1 cut added
M_2 cut added
M_3 cut added
M_4 cut added
M_4 cut added

SDP infeasible

M_1 cut added
M_2 cut added
M_3 cut added
M_4 cut added

SDP infeasible

M_0 cut added
M_1 cut added
M_1 cut added
M_2 cut added
M_2 cut added
M_3 cut added
M_3 cut added
M_4 cut added
M_5 cut added
M_5 cut added
M_5 cut added
M_6 cut added
M_6 cut added
M_6 cut added

SDP infeasible

M_1 cut added
M_3 cut added
M_4 cut added
M_5 cut added
M_5 cut added
M_5 cut added
M_6 cut added
M_6 cut added

SDP infeasible

M_1 cut added
M_2 cut added
M_3 cut added
M_4 cut added

SDP infeasible

M_1 cut added
M_2 cut added
M_4 cut added
M_4 cut added

SDP infeasible


In [182]:
solution

{'status': 'OPTIMAL', 'time': 62.15000033378601, 'cuts': 64}

In [187]:
#feasible_values[-1][f'k[{obj_index}]']

In [184]:
[feasible_values[-1][f'k[{r}]'] for r in range(R)]

[2.3652041100669305e-06,
 6.769397133382133e-07,
 49.999299874781464,
 1.0,
 1.5558997525707238e-06,
 1.5726653156395724e-06,
 1.0712019270557152,
 0.003709451830737228,
 5.03247296613742e-07,
 6.121920558056083e-07,
 89.13454162419283,
 2.741815403355588,
 0.03453148394602389,
 9.560314808047815]

In [ ]:
# wrong: ix, iy error
rho_12 = compute_feasible_correlation(S, d, feasible_values[-1], 0, 1)
rho_13 = compute_feasible_correlation(S, d, feasible_values[-1], 0, 2)
rho_23 = compute_feasible_correlation(S, d, feasible_values[-1], 1, 2)
float(rho_12), float(rho_13), float(rho_23)

(0.3049043972444174, -0.24336720473910886, -0.06531046754732577)

In [186]:
corr = np.corrcoef(sample.T)
float(corr[1, 0]), float(corr[2, 0]), float(corr[2, 1])

(0.32292790498325147, -0.2662006990903024, -0.015608888808065501)

## Notes

- PLN with 2 uncorrelated mRNAs, positive and negative correlation to miRNA is feasible
    - feasible point correlation close match to data
        - note no capture here
        - may be a much wider interval and just lucky point

# Telegraph miRNA - Birth Death mRNA

In [247]:
# settings
Sigma = np.array([
    [1, 0.5, -0.8],
    [0.5, 1, 0],
    [-0.8, 0, 1]]
)
A = np.linalg.cholesky(Sigma)
b = np.array([0.5, 0.5, 0.5])
G = 3

n = 1000
N = 1000

# sample
z = rng.normal(size=(n, G))
lam = np.exp(z @ A.T + b)
sample = rng.poisson(lam)

In [248]:
# settings
k_tx = 5
k_deg = 1
k_reg = 0

n = 1000
N = 1000

# sample
sample_x1 = rng.poisson(k_tx / k_deg, size=n)
sample_x2 = rng.poisson(k_tx / k_deg, size=n)
sample_x3 = rng.poisson(k_tx / k_deg, size=n)
#sample = np.vstack([sample_x1, sample_x2, sample_x3]).T

In [249]:
# downsample
mean_capture = 1.0
downsample, beta = downsample_data(sample, mean_capture)

# mean expression level
print(f"Mean expression {np.mean(downsample)}")

Mean expression 2.703666666666667


In [252]:
# settings
reactions = [
    "1 - xs[3]",
    "xs[3]",
    "xs[3]",
    "xs[0]",
    "1",
    "xs[1]",
    "1",
    "xs[2]",
    "xs[0] * xs[1]",
    "xs[0] * xs[2]"
]
vrs = [
    [0, 0, 0, 1],
    [0, 0, 0, -1],
    [1, 0, 0, 0],
    [-1, 0, 0, 0],
    [0, 1, 0, 0,],
    [0, -1, 0, 0],
    [0, 0, 1, 0],
    [0, 0, -1, 0],
    [-1, -1, 0, 0],
    [-1, 0, -1, 0]
]
db = 2
R = 10
S = 4
U = [3]

d = 3
d_bd = 3
d_me = 3
d_sd = 3

K = 100
fixed = [(3, 1)] # [(3, 1)]
obj_index = -1 # 8

# constraints
constraints = {
    'moment_bounds':           True,
    'moment_matrices':         True,
    'moment_equations':        True,
    'unobserved_constraints':  True,
}

# bootstrap
OB_bounds = bootstrap(downsample, d, N)

# test feasibility
solution, eigenvalues, optim_times, feasible_values = feasibility_test(
    OB_bounds,
    beta,
    reactions,
    vrs,
    db,
    R,
    S,
    U, 
    d,
    d_bd,
    d_me,
    d_sd,
    obj_index,
    constraints,
    fixed=fixed,
    time_limit=300,
    K=K,
    print_evals=False,
    eval_eps=10**-6,
    printing=True,
    silent=True,
    write_model=False,
    load_model=False,
    cut_limit=1000,
    total_time_limit=300
)

SDP infeasible

M_1 cut added
M_2 cut added
M_3 cut added
M_4 cut added

SDP infeasible

M_2 cut added
M_4 cut added
M_4 cut added

SDP infeasible

M_2 cut added
M_3 cut added
M_4 cut added
M_4 cut added

SDP infeasible

M_1 cut added
M_2 cut added
M_3 cut added
M_4 cut added
M_4 cut added

SDP infeasible

M_1 cut added
M_2 cut added
M_3 cut added
M_4 cut added

SDP infeasible

M_2 cut added
M_3 cut added
M_4 cut added
M_4 cut added

SDP infeasible

M_2 cut added
M_2 cut added
M_3 cut added
M_4 cut added

SDP infeasible

M_1 cut added
M_2 cut added
M_4 cut added
M_4 cut added

SDP infeasible

M_1 cut added
M_2 cut added
M_3 cut added
M_4 cut added

SDP infeasible

M_1 cut added
M_2 cut added
M_3 cut added
M_4 cut added
M_4 cut added

SDP infeasible

M_2 cut added
M_3 cut added
M_4 cut added

SDP infeasible

M_1 cut added
M_2 cut added
M_3 cut added
M_4 cut added

SDP infeasible

M_1 cut added
M_2 cut added
M_3 cut added
M_4 cut added

SDP infeasible

M_1 cut added
M_2 cut added
M_3 cut

In [253]:
solution

{'status': 'CUT_LIMIT', 'time': 17.162999629974365, 'cuts': 1000}

In [222]:
#feasible_values[-1][f'k[{obj_index}]']

In [223]:
[feasible_values[-1][f'k[{r}]'] for r in range(R)]

[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0]

In [ ]:
# wrong: ix, iy error
rho_12 = compute_feasible_correlation(S, d, feasible_values[-1], 0, 1)
rho_13 = compute_feasible_correlation(S, d, feasible_values[-1], 0, 2)
rho_23 = compute_feasible_correlation(S, d, feasible_values[-1], 1, 2)
float(rho_12), float(rho_13), float(rho_23)

(0.3526997119451479, -0.35003063302326404, -0.05276234170056024)

In [225]:
corr = np.corrcoef(sample.T)
float(corr[1, 0]), float(corr[2, 0]), float(corr[2, 1])

(0.3292531350538356, -0.2985106039106982, 0.004552870251718321)

## Notes

- PLN also feasible when only miRNA telegraph

# Notes

- any telegraph inclusion i.e. unobserved species, needs many cuts
- optimizing rate needs many more cuts than optimizing 0
- reducing K speedups optimization a lot
    - but concern of cutting off feasible points
    - consider k < K & no fixed idea


# Testing

- try k < K
    - also need to prevent k = 0 e.g. k_r > eps lower bound

In [ ]:
# settings
Sigma = np.array([
    [1, 0.5, -0.8],
    [0.5, 1, 0],
    [-0.8, 0, 1]]
)
A = np.linalg.cholesky(Sigma)
b = np.array([0.5, 0.5, 0.5])
G = 3

n = 1000
N = 1000

# sample
z = rng.normal(size=(n, G))
lam = np.exp(z @ A.T + b)
#sample = rng.poisson(lam)

In [259]:
# settings
k_tx = 5
k_deg = 1
k_reg = 0

n = 1000
N = 1000

# sample
sample_x1 = rng.poisson(k_tx / k_deg, size=n)
sample_x2 = rng.poisson(k_tx / k_deg, size=n)
sample_x3 = rng.poisson(k_tx / k_deg, size=n)
sample = np.vstack([sample_x1, sample_x2, sample_x3]).T

In [260]:
# downsample
mean_capture = 1.0
downsample, beta = downsample_data(sample, mean_capture)

# mean expression level
print(f"Mean expression {np.mean(downsample)}")

Mean expression 4.962


In [270]:
# settings
reactions = [
    "1 - xs[3]",
    "xs[3]",
    "xs[3]",
    "xs[0]",
    "1 - xs[4]",
    "xs[4]",
    "xs[4]",
    "xs[1]",
    "1 - xs[5]",
    "xs[5]",
    "xs[5]",
    "xs[2]",
    "xs[0] * xs[1]",
    "xs[0] * xs[2]"
]
vrs = [
    [0, 0, 0, 1, 0, 0],
    [0, 0, 0, -1, 0, 0],
    [1, 0, 0, 0, 0, 0],
    [-1, 0, 0, 0, 0, 0],
    [0, 0, 0, 0, 1, 0],
    [0, 0, 0, 0, -1, 0],
    [0, 1, 0, 0, 0, 0],
    [0, -1, 0, 0, 0, 0],
    [0, 0, 0, 0, 0, 1],
    [0, 0, 0, 0, 0, -1],
    [0, 0, 1, 0, 0, 0],
    [0, 0, -1, 0, 0, 0],
    [-1, -1, 0, 0, 0, 0],
    [-1, 0, -1, 0, 0, 0]
]
db = 2
R = 14
S = 6
U = [3, 4, 5]

d = 3
d_bd = 3
d_me = 3
d_sd = 3

K = 10
fixed = [] # [(3, 1)]
fixed_weak = [(3, 0.1)]
obj_index = -1 #12

# constraints
constraints = {
    'moment_bounds':           True,
    'moment_matrices':         True,
    'moment_equations':        True,
    'unobserved_constraints':  True,
}

# bootstrap
OB_bounds = bootstrap(downsample, d, N)

# test feasibility
solution, eigenvalues, optim_times, feasible_values = feasibility_test(
    OB_bounds,
    beta,
    reactions,
    vrs,
    db,
    R,
    S,
    U, 
    d,
    d_bd,
    d_me,
    d_sd,
    obj_index,
    constraints,
    fixed=fixed,
    fixed_weak=fixed_weak,
    time_limit=300,
    K=K,
    print_evals=False,
    eval_eps=10**-6,
    printing=True,
    silent=True,
    write_model=False,
    load_model=False,
    cut_limit=100,
    total_time_limit=300
)

SDP infeasible

M_0 cut added
M_1 cut added
M_1 cut added
M_1 cut added
M_2 cut added
M_2 cut added
M_3 cut added
M_3 cut added
M_4 cut added
M_4 cut added
M_5 cut added
M_5 cut added
M_5 cut added
M_6 cut added
M_6 cut added
M_6 cut added

SDP infeasible

M_1 cut added
M_1 cut added
M_2 cut added
M_3 cut added
M_3 cut added
M_4 cut added
M_4 cut added
M_4 cut added
M_5 cut added
M_5 cut added
M_5 cut added
M_6 cut added
M_6 cut added
M_6 cut added

SDP infeasible

M_1 cut added
M_2 cut added
M_3 cut added
M_4 cut added
M_4 cut added
M_4 cut added
M_5 cut added
M_6 cut added

SDP infeasible

M_2 cut added
M_3 cut added
M_4 cut added
M_5 cut added
M_6 cut added

SDP infeasible

M_1 cut added
M_2 cut added
M_4 cut added
M_5 cut added
M_6 cut added

SDP infeasible

M_1 cut added
M_3 cut added
M_4 cut added
M_5 cut added
M_5 cut added
M_6 cut added

SDP infeasible

M_2 cut added
M_3 cut added
M_4 cut added
M_5 cut added
M_5 cut added
M_6 cut added

SDP infeasible

M_1 cut added
M_2 cut add

In [271]:
solution

{'status': 'OPTIMAL', 'time': 21.15899968147278, 'cuts': 73}

In [272]:
#feasible_values[-1][f'k[{obj_index}]']

In [273]:
[feasible_values[-1][f'k[{r}]'] for r in range(R)]

[1.1428216834089588e-06,
 6.945917733627933e-07,
 8.123885913232584,
 1.0440886972144103,
 7.161517381754875e-07,
 7.020337173280851e-07,
 6.619168483824849,
 0.7896080387722184,
 5.538876667827265e-07,
 6.193158581373285e-07,
 1.1895339501559379,
 0.03437329210611328,
 0.017600754597279833,
 0.018968882627721624]

In [ ]:
# wrong: ix, iy error
rho_12 = compute_feasible_correlation(S, d, feasible_values[-1], 0, 1)
rho_13 = compute_feasible_correlation(S, d, feasible_values[-1], 0, 2)
rho_23 = compute_feasible_correlation(S, d, feasible_values[-1], 1, 2)
float(rho_12), float(rho_13), float(rho_23)

(-0.2664946860170229, -0.29398592265559836, 0.0031428898667289432)

In [275]:
corr = np.corrcoef(sample.T)
float(corr[1, 0]), float(corr[2, 0]), float(corr[2, 1])

(0.007788664757559181, -0.01733959098087763, 0.00631037010193043)